# Notebook 05 — Healthy Cohort Downsampling (Bootstrap)

The 1000 Genomes individuals carry far more missense variants (~10,000 each) than TCGA-PAAD tumours carry somatic mutations (median 24). To compare both cohorts on an equal footing, each healthy individual is **downsampled** to a mutation count drawn from the TCGA distribution. This is repeated for 100 bootstrap resamples, which KO5 and KO6 use for subagging.

**Shipped data vs. this notebook**

The analyses read `3_Data/healthy_downsampled_bootstrap.parquet`, which is shipped with the repository. It contains bootstraps 0–99 of the exact resamples used for the reported results (8,466,831 rows; MD5 `ee15b150deede4f4ed56b80eadaf96e8`).

This notebook documents how that file was made and regenerates it with the same method. It does **not** reproduce the shipped file row for row: the original run drew k by *position* from a list of TCGA counts whose order came from an unordered `group_by`, so the draws were not reproducible. This version sorts that list (and fully specifies the variant sort order), so it gives identical output on every run. The output is written to a separate, git-ignored file and never overwrites the shipped data.

## Output schema — `healthy_downsampled_bootstrap_regenerated.parquet`

Same schema as the shipped `healthy_downsampled_bootstrap.parquet`. One row per sampled variant–individual pair per bootstrap.

| Column | Type | Notes |
|--------|------|-------|
| `sample_id` | String | 1000G individual ID |
| `gene_name` | String | Gene label from `merged_tcga_1000G.parquet` |
| `protein_variant` | String | Amino acid substitution (e.g. `G12D`) |
| `am_pathogenicity` | Float64 | AlphaMissense score 0–1; no AM threshold applied |
| `bootstrap_id` | Int16 | Bootstrap index (0–99) |

---
## 1 — Setup

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import polars as pl

RANDOM_SEED     = 42
N_BOOTSTRAPS    = 100     # KO5 / KO6 use bootstraps 0–99
HYPERMUTATOR_TMB = 500    # TCGA patients above this mutation count are excluded from the k distribution

DATA_DIR      = Path('../3_Data')
PARQUET       = DATA_DIR / 'merged_tcga_1000G.parquet'
SHIPPED       = DATA_DIR / 'healthy_downsampled_bootstrap.parquet'
SHIPPED_MD5   = 'ee15b150deede4f4ed56b80eadaf96e8'
OUT_PARQUET   = DATA_DIR / 'healthy_downsampled_bootstrap_regenerated.parquet'

pl.Config.set_tbl_rows(-1)

polars.config.Config

---
## 2 — Load Data

- **1000G**: all AM-annotated germline variants (no AM filter — downstream notebooks apply their own threshold).
- **TCGA**: number of somatic mutations per patient (all rows, no AM filter). The hypermutator (TMB > 500) is excluded, leaving 167 patients whose counts form the sampling distribution for k.

In [2]:
df_1kg = (
    pl.read_parquet(PARQUET, columns=['source', 'sample_id', 'gene_name', 'protein_variant', 'am_pathogenicity'])
    .filter(pl.col('source') == '1000G')
    .filter(pl.col('am_pathogenicity').is_not_null())
    .drop('source')
)

df_tcga_tmb = (
    pl.read_parquet(PARQUET, columns=['source', 'sample_id'])
    .filter(pl.col('source') == 'TCGA')
    .group_by('sample_id')
    .agg(pl.len().alias('tmb'))
)
hypermutator_ids = sorted(df_tcga_tmb.filter(pl.col('tmb') > HYPERMUTATOR_TMB)['sample_id'].to_list())

# Sorted, so rng.choice(tcga_counts) is reproducible (group_by order is not)
tcga_counts = np.sort(
    df_tcga_tmb.filter(pl.col('tmb') <= HYPERMUTATOR_TMB)['tmb'].to_numpy()
)

print(f'1000G variants (AM-annotated) : {len(df_1kg):,}')
print(f'1000G individuals             : {df_1kg["sample_id"].n_unique():,}')
print(f'Hypermutators excluded        : {hypermutator_ids}')
print(f'TCGA count distribution       : n={len(tcga_counts)}  median={np.median(tcga_counts):.0f}  '
      f'range={tcga_counts.min()}–{tcga_counts.max()}')

1000G variants (AM-annotated) : 32,063,253
1000G individuals             : 3,202
Hypermutators excluded        : ['TCGA-IB-7651']
TCGA count distribution       : n=167  median=24  range=1–84


---
## 3 — Bootstrap Downsampling

For each bootstrap `b` (RNG seed `RANDOM_SEED + b`) and each 1000G individual (alphabetical order):

1. Draw **k** from the TCGA count distribution.
2. Sample **min(k, n_variants)** of that individual's variants without replacement.

Variants are sorted by `(sample_id, gene_name, am_pathogenicity, protein_variant)` before sampling, so every row position — and therefore every draw — is fully determined.

In [3]:
df_1kg_sorted = (
    df_1kg
    .sort(['sample_id', 'gene_name', 'am_pathogenicity', 'protein_variant'])
    .with_row_index('_row_idx')
)
indiv_row_idx = {
    sid: np.array(idx)
    for sid, idx in (
        df_1kg_sorted
        .group_by('sample_id')
        .agg(pl.col('_row_idx'))
        .iter_rows()
    )
}
healthy_ids = sorted(indiv_row_idx)

boot_dfs = []
for boot_id in range(N_BOOTSTRAPS):
    rng = np.random.default_rng(RANDOM_SEED + boot_id)
    selected = []
    for sid in healthy_ids:
        all_idx = indiv_row_idx[sid]
        k = min(int(rng.choice(tcga_counts)), len(all_idx))
        if k > 0:
            selected.extend(rng.choice(all_idx, size=k, replace=False).tolist())
    boot_dfs.append(
        df_1kg_sorted
        .filter(pl.col('_row_idx').is_in(pl.Series(selected).implode()))
        .drop('_row_idx')
        .with_columns(pl.lit(boot_id, dtype=pl.Int16).alias('bootstrap_id'))
    )
    if (boot_id + 1) % 10 == 0:
        print(f'  {boot_id + 1:3d}/{N_BOOTSTRAPS}  rows={boot_dfs[-1].height:,}')

df_out = pl.concat(boot_dfs)
print(f'Total rows: {len(df_out):,}')

   10/100  rows=83,804
   20/100  rows=84,370
   30/100  rows=85,363
   40/100  rows=85,456
   50/100  rows=84,612
   60/100  rows=84,940
   70/100  rows=85,316
   80/100  rows=83,106
   90/100  rows=84,692
  100/100  rows=84,483
Total rows: 8,468,511


---
## 4 — Validation

Checks the regenerated table for completeness and compares its size per bootstrap with the shipped file (same method, different random draws — so similar but not identical).

In [4]:
assert df_out['bootstrap_id'].n_unique() == N_BOOTSTRAPS, 'Missing bootstrap IDs'
assert (
    df_out.group_by('bootstrap_id').agg(pl.col('sample_id').n_unique())['sample_id'] == len(healthy_ids)
).all(), 'Not every individual is present in every bootstrap'

rows_new = df_out.group_by('bootstrap_id').agg(pl.len())['len'].to_numpy()
print(f'Regenerated : rows/bootstrap mean={rows_new.mean():,.0f}  std={rows_new.std():,.0f}')

if SHIPPED.exists():
    shipped_md5 = hashlib.md5(SHIPPED.read_bytes()).hexdigest()
    rows_shipped = (
        pl.scan_parquet(SHIPPED).group_by('bootstrap_id').agg(pl.len()).collect()['len'].to_numpy()
    )
    print(f'Shipped     : rows/bootstrap mean={rows_shipped.mean():,.0f}  std={rows_shipped.std():,.0f}')
    print(f'Shipped MD5 : {shipped_md5}  ({"OK" if shipped_md5 == SHIPPED_MD5 else "MISMATCH"})')

Regenerated : rows/bootstrap mean=84,685  std=928
Shipped     : rows/bootstrap mean=84,668  std=839
Shipped MD5 : ee15b150deede4f4ed56b80eadaf96e8  (OK)


---
## 5 — Save

Writes to `healthy_downsampled_bootstrap_regenerated.parquet` (git-ignored). To use it in the analyses instead of the shipped file, point the notebooks' bootstrap path to this file.

In [5]:
df_out.select(['sample_id', 'gene_name', 'protein_variant', 'am_pathogenicity', 'bootstrap_id']) \
      .write_parquet(OUT_PARQUET, compression='zstd', compression_level=19)
print(f'Saved: {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 2**20:.1f} MB)')

Saved: ..\3_Data\healthy_downsampled_bootstrap_regenerated.parquet  (56.3 MB)
